In [ ]:
import numpy as np
import math
import trimesh
import plotly.graph_objects as go
import Grasping_Face.grasping as gf
import Grasping_Face.visualize_grasping as gfv

In [ ]:
name = 'obj_000003' # logitech_c930e_m / obj_000001

target_mesh_file = f'./models_target/models_cad/{name}.obj'

In [ ]:
result = gf.compute_best_patch_pairs(
    mesh_path=target_mesh_file,
    mesh_max_triangles = 1000,
    angle_deg=30,              # 패치 병합 허용 각도 (↑면 패치 수 ↓)
    coplanar_tol=1,            # 공면성 허용 오차 (↑면 패치 수 ↓)
    min_opening=10.0,          # 그리퍼 최소 개구(mm)
    max_opening=140.0,         # 그리퍼 최대 개구(mm)
    angle_tolerance_deg=15,    # 패치 페어 정반대 threshold
    top_k=100                  # 상위 패치 페어 후보 개수
)

reports = gf.check_gripper_feasibility_with_yaw(result)

In [ ]:
len(result['top_k'])

In [ ]:
fig = gfv.visualize_merged_patches_plotly(result, show=True)
# fig.write_html(f"./Grasping_Result/{name}_patch_normal.html", include_plotlyjs="cdn", full_html=True)

In [ ]:
fig = gfv.visualize_pairs_centroid_lines(result, show=True)
# fig.write_html(f"./Grasping_Result/{name}_patch_pairs.html", include_plotlyjs="cdn", full_html=True)

In [ ]:
fig = gfv.visualize_feasible_pairs_pads(result, reports, show=True)
# fig.write_html(f"./Grasping_Result/{name}_patch_pairs_feasible.html", include_plotlyjs="cdn", full_html=True)

In [ ]:
import os
import numpy as np
import math
import trimesh
import plotly.graph_objects as go
import Grasping_Face.grasping as gf
import Grasping_Face.visualize_grasping as gfv

cads = [file for file in os.listdir('./models_target/models_cad/') if file.endswith('.obj')]

for cad in cads:
    name = cad.split('.')[0]
    target_mesh_file = f'./models_target/models_cad/{name}.obj'


    result = gf.compute_best_patch_pairs(
        mesh_path=target_mesh_file,
        mesh_max_triangles = 1000,
        angle_deg=30,             # 패치 병합 허용 각도 (↑면 패치 수 ↓)
        coplanar_tol=1,           # 공면성 허용 오차 (↑면 패치 수 ↓)
        min_opening=5.0,          # 그리퍼 최소 개구(mm)
        max_opening=140.0,        # 그리퍼 최대 개구(mm)
        angle_tolerance_deg=15,   # 패치 페어 정반대 threshold
        top_k=100                 # 상위 패치 페어 후보 개수
    )

    # reports = gf.check_gripper_feasibility(result)
    reports = gf.check_gripper_feasibility_with_yaw(result)


    feasible_p = result['top_k']
    feasible_r = [r for r in reports if r.get('feasible')]
    print(f'{name} : patch pairs {len(feasible_p)}, feasible sol. {len(feasible_r)}')

    fig = gfv.visualize_merged_patches_plotly(result)
    fig.write_html(f"./Grasping_Result/3-patch_normal_{name}.html", include_plotlyjs="cdn", full_html=True)

    fig = gfv.visualize_pairs_centroid_lines(result)
    fig.write_html(f"./Grasping_Result/2-patch_pairs_{name}.html", include_plotlyjs="cdn", full_html=True)

    try:
        fig = gfv.visualize_feasible_pairs_pads(result, reports)
        fig.write_html(f"./Grasping_Result/1-patch_pairs_feasible_{name}.html", include_plotlyjs="cdn", full_html=True)
    except:
        pass

#### Test

In [ ]:
mesh_path = target_mesh_file
mesh_max_triangles = 1000
angle_deg = 30
coplanar_tol = 1
min_opening = 10.0
max_opening = 140.0
angle_tolerance_deg = 15
top_k = 100

In [ ]:
remesh, mesh_quad = gf.load_uniform_mesh_with_open3d(mesh_path, target_triangles=mesh_max_triangles)

In [ ]:
patches = gf.extract_planar_patches(remesh, angle_deg=angle_deg, coplanar_tol=coplanar_tol)
patches = gf.orient_patch_normals(mesh_quad, patches, inward=False)  # false: 모두 바깥쪽으로 정렬

len(patches)

In [ ]:
params = gf.PatchPairParams(
    min_opening=min_opening,
    max_opening=max_opening,
    angle_tolerance_deg=angle_tolerance_deg,
)

In [ ]:
np.linalg.norm(patches[70].centroid)

In [ ]:
patches[70].centroid

In [ ]:
cand = gf.score_patch_pair(patches[70], patches[74], params, remesh)
cand

In [ ]:
cands = []
nvecs = [gf.unit(np.array(p.normal)) for p in patches] # unit normals 
# 현재 패치와 normals가 angle_tolerance_deg 이하인 patch만 남김 (각도 180도 +- angle_tolerance_deg 범위)
for i in range(len(patches) - 1):
    ni = nvecs[i]

    best_cand = None
    best_score = -1.0

    for j in range(i + 1, len(patches)):
        # i-j 법선 각도
        dot = float(ni @ nvecs[j])
        ang = math.degrees(math.acos(max(-1.0, min(1.0, dot))))
        # 180° - tol 보다 작으면 충분히 반대가 아님 -> 스킵
        if ang < 180.0 - angle_tolerance_deg:
            continue

        cand = gf.score_patch_pair(patches[i], patches[j], params, remesh)

        # 기존 폭/면적/점수 필터
        if cand.width < params.min_opening or cand.width > params.max_opening:
            continue
        if cand.score <= 0.0:
            continue

        print(i,j)


        # i에 대해 최고 점수만 유지
        if cand.score > best_score:
            best_score = cand.score
            best_cand = cand

    if best_cand is not None:
        cands.append(best_cand)

cands.sort(key=lambda c: c.score, reverse=True)

In [ ]:
len(cands)